In [1]:
import pandas as pd
import cv2
import numpy as np
import os
from tqdm import tqdm

path = "/kaggle/input/ham1000-segmentation-and-classification"

# Folders
IMAGE_DIR = os.path.join(path, "images")
MASK_DIR = os.path.join(path, "masks")
CSV_PATH = os.path.join(path, "GroundTruth.csv")

In [2]:
# !pip install opencv-python pandas numpy matplotlib

In [3]:
# 1. Configuration

IMAGE_ID_COLUMN = 'image'  
LABEL_COLUMNS = ['MEL', 'NV', 'BCC','AKIEC','BKL','DF','VASC']
TARGET_SIZE = (224, 224)

In [4]:
# 2. Pre-processing Functions

def remove_hair(image):
    """Removes hair from a skin image using morphological operations."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) #convert to grayscale
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15)) #15*15 kernel
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel) #using blackhat morphology for hair detection
    _, hair_mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)#hair masked over intensity 10
    inpainted_image = cv2.inpaint(image, hair_mask, 3, cv2.INPAINT_TELEA)#inpaint the mask
    return inpainted_image

def enhance_image(image):
    """Enhances contrast of the skin image using CLAHE."""
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)  # convert to LAB color space
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))  # CLAHE
    cl = clahe.apply(l)  # apply to L-channel
    enhanced_lab = cv2.merge((cl, a, b))
    enhanced_image = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)
    return enhanced_image

def preprocess_for_cnn(image_path, mask_path, target_size):
    """Full preprocessing pipeline for one image."""
    image = cv2.imread(image_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if image is None or mask is None:
        raise ValueError("Image or mask not found.")

    # Hair removal
    image_no_hair = remove_hair(image)

    # Image enhancement
    enhanced_image = enhance_image(image_no_hair)
    
    # Binary mask
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # Apply mask
    masked_image = cv2.bitwise_and(enhanced_image, enhanced_image, mask=binary_mask)


    # Crop to ROI (if mask is valid)
    if cv2.countNonZero(binary_mask) > 0:
        x, y, w, h = cv2.boundingRect(binary_mask)
        cropped_image = masked_image[y:y+h, x:x+w]
    else:
        cropped_image = image_no_hair  # fallback: use full image

    # Resize + normalize
    resized_image = cv2.resize(cropped_image, target_size)
    normalized_image = resized_image.astype(np.float32) / 255.0

    return normalized_image


In [5]:
# 3. Data Loading

print("Loading data from CSV...")
df = pd.read_csv(CSV_PATH)

# Add file paths
def find_image_path(img_id):
    for ext in ['.jpg', '.png', '.jpeg']:
        path = os.path.join(IMAGE_DIR, img_id + ext)
        if os.path.exists(path):
            return path
    return None

def find_mask_path(img_id):
    for ext in ['.png', '.jpg']:
        path = os.path.join(MASK_DIR, img_id + '_segmentation' + ext)
        if os.path.exists(path):
            return path
    return None


df['image_path'] = df[IMAGE_ID_COLUMN].apply(find_image_path)
df['mask_path'] = df[IMAGE_ID_COLUMN].apply(find_mask_path)

# Extract labels
all_labels = df[LABEL_COLUMNS].to_numpy()
print(f"Extracted labels with shape: {all_labels.shape}")


Loading data from CSV...
Extracted labels with shape: (10015, 7)


In [ ]:
# 4. Pre-process Images

processed_images = []
valid_indices = []

print("\nStarting image preprocessing...")
for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    try:
        if row['image_path'] and row['mask_path']:
            processed_img = preprocess_for_cnn(row['image_path'], row['mask_path'], TARGET_SIZE)
            processed_images.append(processed_img)
            valid_indices.append(index)
    except Exception as e:
        print(f"Error at index {index} ({row[IMAGE_ID_COLUMN]}): {e}")

# Convert to arrays
X = np.array(processed_images, dtype=np.float32)
y = all_labels[valid_indices]

print("\nProcessing Complete")
print(f"Processed {len(X)} images.")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


print(df[['image', 'image_path', 'mask_path']].head(5))


print(X.shape)
print(y.shape)


Starting image preprocessing...


 81%|████████  | 8130/10015 [15:29<03:12,  9.77it/s]

In [ ]:
from sklearn.model_selection import train_test_split

# One-hot → single label for stratification
y_single = np.argmax(y, axis=1)

# First split train+val vs test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.1, random_state=12, stratify=y_single
)

# Split train vs validation
y_single_trainval = np.argmax(y_trainval, axis=1)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=12, stratify=y_single_trainval
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Input, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

# tf.keras.backend.clear_session()   # release old graphs

# # Let TensorFlow grow GPU memory instead of preallocating everything
# gpus = tf.config.list_physical_devices('GPU')
# if gpus:
#     try:
#         for gpu in gpus:
#             tf.config.experimental.set_memory_growth(gpu, True)
#     except Exception as e:
#         print("Could not set memory growth:", e)

num_classes = y.shape[1]

model = Sequential([
    # Block 1
    Input(shape=(224,224,3)),
    Conv2D(32, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    # Block 2
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    # Block 3
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),
    
    # Flatten + Dense
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    
    # Output Layer
    Dense(num_classes, activation='softmax')
])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7, verbose=1),
    ModelCheckpoint("best_model.h5", monitor='val_loss', save_best_only=True, verbose=1)
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=32,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
loss, acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {acc:.4f}")

In [ ]:
# Evaluate on Test Set
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=1)
print(f"✅ Test Accuracy: {test_acc:.4f}")

# Plot curves
import matplotlib.pyplot as plt
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title("Accuracy over Epochs")
plt.xlabel("Epochs"); plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("Loss over Epochs")
plt.xlabel("Epochs"); plt.ylabel("Loss")
plt.legend()

plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Predict on test set
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

# Classification report
print("Classification Report:\n")
print(classification_report(y_true, y_pred_classes))